In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting to verify
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

gen_details = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv')
hw_tseries = pd.read_csv('/scratch/ng72/ms5578/time_series/gen_hw_status.csv')
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
# Select start and end dates (Jul 2009 - Jun 2024)
sdate, edate = '2009-07-01','2024-06-30'

In [4]:
# Select frequency of timeseries (hourly or daily)
mode = 'hourly'

In [5]:
# Select region or fuel type (optional)

def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp

info = select_group(gen_details,ftype=['Wind','Solar']).copy()

The functions below retrieve the timeseries for the specified dates.
The data is resampled to daily if specified above. The df is merged with the generation information.

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None, mode='daily'):
    import os
    import pandas as pd
    import numpy as np

    # Sanitize DUIDs and build file paths
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp, dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        raise ValueError("no files to load.")

    # Concatenate and clean header rows
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # Type conversions
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)

    # Filter by date
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time']).dt.normalize()
    hw_tseries = hw_tseries.set_index('time').sort_index()
    hw_tseries = hw_tseries.loc[start_date:end_date]

    if mode == 'daily':
        # Aggregate dfs to daily
        agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'max'}
        dfs_daily = dfs.groupby(['DUID', pd.Grouper(freq='1D')]).agg(agg_func).reset_index()
        hw_tseries_daily = hw_tseries.reset_index()
        # Normalize time columns to midnight for exact matching
        dfs_daily['time'] = pd.to_datetime(dfs_daily['time']).dt.normalize()
        hw_tseries_daily['time'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge on DUID and time
        merged = pd.merge(
            dfs_daily,
            hw_tseries_daily,
            on=['DUID', 'time'],
            how='left'
        )
    elif mode == 'hourly':
        dfs_hourly = dfs.reset_index()
        # Create full hourly time index for each DUID
        duids = dfs_hourly['DUID'].unique()
        hourly_range = pd.date_range(start=start_date, end=end_date, freq='1h')
        full_index = pd.MultiIndex.from_product([duids, hourly_range], names=['DUID', 'time'])
        hw_hourly = pd.DataFrame(index=full_index).reset_index()
        hw_hourly['date'] = hw_hourly['time'].dt.normalize()
        # Prepare daily hw_tseries for merging
        hw_tseries_daily = hw_tseries.reset_index()
        hw_tseries_daily['date'] = pd.to_datetime(hw_tseries_daily['time']).dt.normalize()
        # Merge daily values onto hourly DataFrame by DUID and date
        hw_tseries_broadcast = pd.merge(
            hw_hourly,
            hw_tseries_daily.drop(columns='time'),
            on=['DUID', 'date'],
            how='left'
        ).drop(columns='date')
        # Merge on DUID and time
        merged = pd.merge(
            dfs_hourly,
            hw_tseries_broadcast,
            on=['DUID', 'time'],
            how='left'
        )
    else:
        raise ValueError("mode must be 'daily' or 'hourly'")

    merged = merged.dropna(how='all')

    # Jitter coordinates for plotting
    def jitter(group):
        duplicates = group.groupby(['lat', 'lon']).cumcount()
        np.random.seed(42)
        jitter_strength = 0.05
        group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
        group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
        return group

    grp = jitter(grp)

    df = merged.merge(
        grp[['DUID', 'fuel_source_primary', 'region', 'lat_jittered', 'lon_jittered']],
        on='DUID',
        how='left'
    )

    return df.reset_index(drop=True)

In [ ]:
# Make sure mode is set correctly to daily or hourly
df = process_group(info, gen_fpath, hw_tseries, sdate, edate, mode=mode)
df

The following section contains all the data cleaning functions and their execution. These should be selected based on technology type and research question.

These can be used for almost all code

In [8]:
def sel_months(df, months=[12,1,2]):
    df = df[df['time'].dt.month.isin(months)]
    return df

In [13]:
def min_heatwave_days(df, min_days=20):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Works with hourly or higher-frequency data by counting unique *days*.

    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    
    # Normalize to midnight → one unique timestamp per day
    df['date'] = df['time'].dt.normalize()

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [9]:
def remove_negatives(df):
    # Removes negatives from df, generally fine to use for all except hydro (which is outside scope).
    df = df[~((df['TOTALMWh'] < 0))]
    return df

These can be used for hourly data

In [10]:
def remove_solar_night(df):
    # For solar only: remove rows between 20:00 and 06:00
    solar_mask = df['fuel_source_primary'] == 'Solar'
    times = df.loc[solar_mask, 'time'].dt.time
    
    time_filter = ~(
        (times >= datetime.time(21, 0)) |  # 21:00 onwards
        (times < datetime.time(5, 0))      # before 05:00
    )
    
    # Keep all non-solar rows, and solar rows passing the time filter
    df = pd.concat([
        df[~solar_mask],
        df.loc[solar_mask].loc[time_filter]
    ])

    return df

In [11]:
def remove_wind_zeros(df, duid_col='DUID', value_col='TOTALMWh',
                      tech_col='fuel_source_primary', wind_label='Wind', threshold=None):
    # Mann-Whitney U tests are sensitive to excessive zeros, so any DUIDs with over 40% 0s are excluded from the data.
    """
    Remove *all rows* for wind DUIDs where the percentage of zeros 
    exceeds the given threshold. Non-wind DUIDs are left untouched.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        duid_col (str): Column name for grouping (default 'DUID')
        value_col (str): Column name with numeric values (default 'TOTALMWh')
        tech_col (str): Column that identifies technology type (default 'fuel_source_primary')
        wind_label (str): Label used for wind in tech_col (default 'Wind')
        threshold (float): Maximum allowed percentage of zeros (default 5)
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    
    # work only on wind rows
    wind_df = df[df[tech_col] == wind_label]
    
    # calculate % of zeros per wind DUID
    percent_zeros = (
        wind_df.groupby(duid_col)[value_col]
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    
    # keep DUIDs below threshold
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    
    # filter wind df
    wind_filtered = wind_df[wind_df[duid_col].isin(keep_duids)]
    
    # keep all non-wind rows
    non_wind_df = df[df[tech_col] != wind_label]
    
    # combine and return
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

These can be used for specific tech types

In [12]:
def clear_agc(df):
    # Removes days where AGCSTATUS is 0 and the plant is non-operational.
    # Generally ill-advised
    df = df[~((df['TOTALMWh'] < 0))]
    mask = (df['fuel_source_primary'].isin([
            'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
            'Brown Coal', 'Diesel', 'Kerosene'
        ]) & (df['TOTALCLEARED'] <= 0))
    df.loc[mask, 'TOTALMWh'] = np.nan
    return df

In [14]:
df = sel_months(df)
df = remove_negatives(df)
df = remove_wind_zeros(df, threshold=40)
df = min_heatwave_days(df,20)

In [ ]:
df